## Calculating Venus "OR/Target" attitudes from given PCAD/ACA attitudes

In [2]:
from astropy.table import Table
from Quaternion import Quat
import chandra_aca

These are the attitudes from Eric's original '2017_Venus_Obs_Summary.txt' with the new obsids used as the labels and other columns cut.

In [3]:
PCAD_atts = """
Obs     Start_Time    End_Time        q1          q2           q3           q4
17439   009:18:21:00  009:20:05:30    -0.54124474  0.17440748  -0.10503936  0.81584490
18690   009:20:09:30  009:21:54:00    -0.54128440  0.17382505  -0.10475117  0.81597993
18691   009:21:58:00  009:23:42:30    -0.54133029  0.17320695  -0.10443884  0.81612095
18692   009:23:46:30  010:01:31:00    -0.54131932  0.17257327  -0.10423103  0.81628902
18693   010:01:35:00  010:03:19:30    -0.54134818  0.17195804  -0.10395069  0.81643544
18694   010:03:23:30  010:05:08:00    -0.54136894  0.17131557  -0.10367493  0.81659179
18695   010:05:12:00  010:06:56:30    -0.54137633  0.17071488  -0.10344604  0.81674171
18696   010:07:00:30  010:08:45:00    -0.54140053  0.17002964  -0.10314792  0.81690630
"""

In [4]:
t = Table.read(PCAD_atts, format='ascii')

This is the "original"/"used for most of the mission" ODB_SI_ALIGN transform matrix.  I think it is also the reference matrix being used in SAUSAGE.  However, OFLS is using the 07JUL16 characteristics version, so I'm not 100% that this is the right matrix.  However, if I take the PCAD attitudes in pre-products, and invert them by this matrix, I end up with the current OR-list attitudes (off by the dynamic offsets).

In [5]:
ODB_SI_ALIGN = chandra_aca.transform.ODB_SI_ALIGN

In [6]:
ODB_SI_ALIGN

array([[  9.99999906e-01,  -3.37419984e-04,  -2.73439987e-04],
       [  3.37419984e-04,   9.99999943e-01,  -4.61320600e-08],
       [  2.73439987e-04,  -4.61320600e-08,   9.99999963e-01]])

Here, for each of these obsids, I've converted from ACA to "target" coordinates for each, added each attitude to a data structure and then just printed the thing.  The full code for calc_targ_from_aca is just the multiplication in: https://github.com/sot/chandra_aca/blob/ca0442f35b35da208d9ddb64f4db911e95a42ea6/chandra_aca/transform.py#L159

In [7]:
targ_atts = []
for obs in t:
    q_aca = Quat([obs['q1'], obs['q2'], obs['q3'], obs['q4']])
    q_targ = chandra_aca.calc_targ_from_aca(q_aca, y_off=0, z_off=0, si_align=ODB_SI_ALIGN)
    targ_atts.append({'obsid': obs['Obs'],
                     'q1': q_targ.q[0], 'q2': q_targ.q[1], 'q3': q_targ.q[2], 'q4': q_targ.q[3],
                     'ra': q_targ.ra, 'dec': q_targ.dec, 'roll': q_targ.roll})
targ_atts = Table(targ_atts)

In [8]:
targ_atts[['obsid', 'q1', 'q2', 'q3', 'q4', 'ra', 'dec', 'roll']]

obsid,q1,q2,q3,q4,ra,dec,roll
int64,float64,float64,float64,float64,float64,float64,float64
17439,-0.541229663934,0.174387246974,-0.104827717354,0.815886446944,338.579685658,-9.85112103621,291.014641357
18690,-0.541269382794,0.173804805217,-0.104539499158,0.816021348691,338.648173371,-9.81624028852,291.032228435
18691,-0.541315334371,0.173186693694,-0.1042271391,0.816162231488,338.721400868,-9.77952690438,291.050314569
18692,-0.541304442867,0.172552988879,-0.104019302249,0.816330179787,338.786324325,-9.7359722092,291.076198486
18693,-0.541333368334,0.171937743744,-0.103738933608,0.816476468373,338.856403186,-9.69780114657,291.096184493
18694,-0.541354199023,0.171295255886,-0.103463144398,0.816632684008,338.928024491,-9.65703625386,291.118169435
18695,-0.541361659071,0.170694546649,-0.1032342281,0.816782483262,338.99231415,-9.61730422493,291.140155315
18696,-0.541385933918,0.170009288246,-0.102936077031,0.816946929276,339.068981826,-9.57397355144,291.163139357


## If we want to pre-correct for dynamic offsets, we can use the offsets in the pre-products dynamic offset table. 

It also looks like all of these dynamic offsets are small.  I think these would be larger if the observations were handled as cycle 18 observations (presently being handled as cycle 16 observations, which seems fine).

In [9]:
dynamic_offsets = Table.read("""
obsid detector chipx chipy chip_id target_offset_y target_offset_z target_ra target_dec target_roll aca_offset_y aca_offset_z aca_ra aca_dec aca_roll mean_date mean_t_ccd
17764  ACIS-S  200.7  476.9  7  0.00  -18.00  76.956542  30.401439  262.690000  7.20  0.72  76.947191  30.423980  262.694735  2017:009:12:20:15.816  -13.25
17439  ACIS-I  932.0  1009.0  3  0.00  0.00  338.557806  -9.838692  291.010900  6.69  -0.71  338.535439  -9.824457  291.007081  2017:009:19:10:15.816  -13.58
18690  ACIS-I  932.0  1009.0  3  0.00  0.00  338.626292  -9.803817  291.028500  7.00  -0.56  338.603851  -9.789523  291.024682  2017:009:20:59:35.816  -13.66
18691  ACIS-I  932.0  1009.0  3  0.00  0.00  338.699513  -9.767111  291.046600  7.21  -0.45  338.677022  -9.752780  291.042788  2017:009:22:48:55.816  -13.71
18692  ACIS-I  932.0  1009.0  3  0.00  0.00  338.764439  -9.723566  291.072500  7.34  -0.39  338.741913  -9.709217  291.068699  2017:010:00:38:15.816  -13.75
18693  ACIS-I  932.0  1009.0  3  0.00  0.00  338.834516  -9.685402  291.092500  7.42  -0.35  338.811971  -9.671045  291.088710  2017:010:02:27:35.816  -13.77
18694  ACIS-I  932.0  1009.0  3  0.00  0.00  338.906135  -9.644646  291.114500  7.44  -0.34  338.883580  -9.630291  291.110724  2017:010:04:16:55.816  -13.77
18695  ACIS-I  932.0  1009.0  3  0.00  0.00  338.970422  -9.604922  291.136500  7.44  -0.34  338.947866  -9.590577  291.132740  2017:010:06:03:31.816  -13.77
18696  ACIS-I  932.0  1009.0  3  0.00  0.00  339.047088  -9.561600  291.159500  7.41  -0.36  339.024536  -9.547271  291.155757  2017:010:07:44:39.816  -13.76""", format='ascii')

## OR List

For corrected target attitudes, if the current process demands that SAUSAGE apply the dynamic aimpoint correction, I think we can either use the attitudes in targ_atts_dyn that have been corrected both for SI align and an estimate of the aca dynamic offset *OR* we can use the attitudes that don't include that dynamic correction, and handle the correction in the TARGET_OFFSETs.  I think I'll go with the second method for slightly more flexibility:

In [10]:
orig_OR = """
BEGIN_COMMENT, ID=17439
Window Constraint requirements exist for observation.
  WINDOW=(2017:007:00:00:00,2017:012:23:59:00)
Remarks:
| observing period: 2015 Oct 16 - Nov 07, with preference to Oct 16 - 
20 | instrument: ACIS-I(Venus is optically too bright for ACIS-S) | active 
CCDs: 
only I1 and I3 | readout mode: half array | boresight: Venus should be kept 
within the 1' HEW circle | exposure: 6 x 8 ks = 48 ks 
Note: Current coordinates of 333.395208, -10.236917 are nominal, just to allow 
placement in LTS. (DJP 10/21/16)
END_COMMENT
OBS,
 ID=17439,TARGET=(338.557806,-9.838692,{Venus}),DURATION=(6000.000000),
 PRIORITY=4,SI=ACIS-I,GRATING=NONE,SI_MODE=TE_00C3E,ACA_MODE=DEFAULT,
 TARGET_OFFSET=(0.000000,0.000000),
 PLANET=(VENUS,OFF),
 DITHER=(ON,0.002222,0.360000,0.000000,0.002222,0.509100,0.000000),
 WINDOW=(2017:009:18:24:47.008,2017:012:23:59:00.000),
 ROLL=(291.010900,0.000000),SEGMENT=(1,5700.000000),PRECEDING=(17764),
 MIN_ACQ=1,MIN_GUIDE=1

BEGIN_COMMENT, ID=18690
Window Constraint requirements exist for observation.
  WINDOW=(2017:007:00:00:00,2017:012:23:59:00)
Remarks:
| observing period: 2015 Oct 16 - Nov 07, with preference to Oct 16 - 
20 | instrument: ACIS-I(Venus is optically too bright for ACIS-S) | active 
CCDs: 
only I1 and I3 | readout mode: half array | boresight: Venus should be kept 
within the 1' HEW circle | exposure: 6 x 8 ks = 48 ks 
Note: Current coordinates of 338.395208, -10.236917 are nominal, just to allow 
placement in LTS. (DJP 10/21/16)
END_COMMENT
OBS,
 ID=18690,TARGET=(338.626292,-9.803817,{Venus}),DURATION=(6000.000000),
 PRIORITY=4,SI=ACIS-I,GRATING=NONE,SI_MODE=TE_00C3E,ACA_MODE=DEFAULT,
 TARGET_OFFSET=(0.000000,0.000000),
 PLANET=(VENUS,OFF),
 DITHER=(ON,0.002222,0.360000,0.000000,0.002222,0.509100,0.000000),
 WINDOW=(2017:007:00:00:00.000,2017:012:23:59:00.000),
 ROLL=(291.028500,0.000000),SEGMENT=(1,5700.000000),PRECEDING=(17439),
 MIN_ACQ=1,MIN_GUIDE=1

BEGIN_COMMENT, ID=18691
Window Constraint requirements exist for observation.
  WINDOW=(2017:007:00:00:00,2017:012:23:59:00)
Remarks:
| observing period: 2015 Oct 16 - Nov 07, with preference to Oct 16 - 
20 | instrument: ACIS-I(Venus is optically too bright for ACIS-S) | active 
CCDs: 
only I1 and I3 | readout mode: half array | boresight: Venus should be kept 
within the 1' HEW circle | exposure: 6 x 8 ks = 48 ks 
Note: Current coordinates of 338.395208, -10.236917 are nominal, just to allow 
placement in LTS. (DJP 10/21/16)
END_COMMENT
OBS,
 ID=18691,TARGET=(338.699513,-9.767111,{Venus}),DURATION=(6000.000000),
 PRIORITY=4,SI=ACIS-I,GRATING=NONE,SI_MODE=TE_00C3E,ACA_MODE=DEFAULT,
 TARGET_OFFSET=(0.000000,0.000000),
 PLANET=(VENUS,OFF),
 DITHER=(ON,0.002222,0.360000,0.000000,0.002222,0.509100,0.000000),
 WINDOW=(2017:007:00:00:00.000,2017:012:23:59:00.000),
 ROLL=(291.046600,0.000000),SEGMENT=(1,5700.000000),PRECEDING=(18690),
 MIN_ACQ=1,MIN_GUIDE=1

BEGIN_COMMENT, ID=18692
Window Constraint requirements exist for observation.
  WINDOW=(2017:007:00:00:00,2017:012:23:59:00)
Remarks:
| observing period: 2015 Oct 16 - Nov 07, with preference to Oct 16 - 
20 | instrument: ACIS-I(Venus is optically too bright for ACIS-S) | active 
CCDs: 
only I1 and I3 | readout mode: half array | boresight: Venus should be kept 
within the 1' HEW circle | exposure: 6 x 8 ks = 48 ks 
Note: Current coordinates of 338.395208, -10.236917 are nominal, just to allow 
placement in LTS. (DJP 10/21/16)
END_COMMENT
OBS,
 ID=18692,TARGET=(338.764439,-9.723566,{Venus}),DURATION=(6000.000000),
 PRIORITY=4,SI=ACIS-I,GRATING=NONE,SI_MODE=TE_00C3E,ACA_MODE=DEFAULT,
 TARGET_OFFSET=(0.000000,0.000000),
 PLANET=(VENUS,OFF),
 DITHER=(ON,0.002222,0.360000,0.000000,0.002222,0.509100,0.000000),
 WINDOW=(2017:007:00:00:00.000,2017:012:23:59:00.000),
 ROLL=(291.072500,0.000000),SEGMENT=(1,5700.000000),PRECEDING=(18691),
 MIN_ACQ=1,MIN_GUIDE=1

BEGIN_COMMENT, ID=18693
Window Constraint requirements exist for observation.
  WINDOW=(2017:007:00:00:00,2017:012:23:59:00)
Remarks:
| observing period: 2015 Oct 16 - Nov 07, with preference to Oct 16 - 
20 | instrument: ACIS-I(Venus is optically too bright for ACIS-S) | active 
CCDs: 
only I1 and I3 | readout mode: half array | boresight: Venus should be kept 
within the 1' HEW circle | exposure: 6 x 8 ks = 48 ks 
Note: Current coordinates of 338.395208, -10.236917 are nominal, just to allow 
placement in LTS. (DJP 10/21/16)
END_COMMENT
OBS,
 ID=18693,TARGET=(338.834516,-9.685402,{Venus}),DURATION=(6000.000000),
 PRIORITY=4,SI=ACIS-I,GRATING=NONE,SI_MODE=TE_00C3E,ACA_MODE=DEFAULT,
 TARGET_OFFSET=(0.000000,0.000000),
 PLANET=(VENUS,OFF),
 DITHER=(ON,0.002222,0.360000,0.000000,0.002222,0.509100,0.000000),
 WINDOW=(2017:007:00:00:00.000,2017:012:23:59:00.000),
 ROLL=(291.092500,0.000000),SEGMENT=(1,5700.000000),PRECEDING=(18692),
 MIN_ACQ=1,MIN_GUIDE=1

BEGIN_COMMENT, ID=18694
Window Constraint requirements exist for observation.
  WINDOW=(2017:007:00:00:00,2017:012:23:59:00)
Remarks:
| observing period: 2015 Oct 16 - Nov 07, with preference to Oct 16 - 
20 | instrument: ACIS-I(Venus is optically too bright for ACIS-S) | active 
CCDs: 
only I1 and I3 | readout mode: half array | boresight: Venus should be kept 
within the 1' HEW circle | exposure: 6 x 8 ks = 48 ks 
Note: Current coordinates of 338.395208 , -10.236917 are nominal, just to allow 
placement in LTS. (DJP 10/21/16)
END_COMMENT
OBS,
 ID=18694,TARGET=(338.906135,-9.644646,{Venus}),DURATION=(6000.000000),
 PRIORITY=4,SI=ACIS-I,GRATING=NONE,SI_MODE=TE_00C3E,ACA_MODE=DEFAULT,
 TARGET_OFFSET=(0.000000,0.000000),
 PLANET=(VENUS,OFF),
 DITHER=(ON,0.002222,0.360000,0.000000,0.002222,0.509100,0.000000),
 WINDOW=(2017:007:00:00:00.000,2017:012:23:59:00.000),
 ROLL=(291.114500,0.000000),SEGMENT=(1,5700.000000),PRECEDING=(18693),
 MIN_ACQ=1,MIN_GUIDE=1

BEGIN_COMMENT, ID=18695
Window Constraint requirements exist for observation.
  WINDOW=(2017:007:00:00:00,2017:012:23:59:00)
Remarks:
| observing period: 2015 Oct 16 - Nov 07, with preference to Oct 16 - 
20 | instrument: ACIS-I(Venus is optically too bright for ACIS-S) | active 
CCDs: 
only I1 and I3 | readout mode: half array | boresight: Venus should be kept 
within the 1' HEW circle | exposure: 6 x 8 ks = 48 ks 
Note: Current coordinates of 338.395208, -10.236917 are nominal, just to allow 
placement in LTS. (DJP 10/21/16)
END_COMMENT
OBS,
 ID=18695,TARGET=(338.970422,-9.604922,{Venus}),DURATION=(6000.000000),
 PRIORITY=4,SI=ACIS-I,GRATING=NONE,SI_MODE=TE_00C3E,ACA_MODE=DEFAULT,
 TARGET_OFFSET=(0.000000,0.000000),
 PLANET=(VENUS,OFF),
 DITHER=(ON,0.002222,0.360000,0.000000,0.002222,0.509100,0.000000),
 WINDOW=(2017:007:00:00:00.000,2017:012:23:59:00.000),
 ROLL=(291.136500,0.000000),SEGMENT=(1,5700.000000),PRECEDING=(18694),
 MIN_ACQ=1,MIN_GUIDE=1

BEGIN_COMMENT, ID=18696
Window Constraint requirements exist for observation.
  WINDOW=(2017:007:00:00:00,2017:012:23:59:00)
Remarks:
| observing period: 2015 Oct 16 - Nov 07, with preference to Oct 16 - 
20 | instrument: ACIS-I(Venus is optically too bright for ACIS-S) | active 
CCDs: 
only I1 and I3 | readout mode: half array | boresight: Venus should be kept 
within the 1" HEW circle | exposure: 6 x 8 ks = 48 ks 
Note: Current coordinates of 338.395208 , -10.236917 are nominal, just to allow 
placement in LTS. (DJP 10/21/16)
END_COMMENT
OBS,
 ID=18696,TARGET=(339.047088,-9.561600,{Venus}),DURATION=(5300.000000),
 PRIORITY=4,SI=ACIS-I,GRATING=NONE,SI_MODE=TE_00C3E,ACA_MODE=DEFAULT,
 TARGET_OFFSET=(0.000000,0.000000),
 PLANET=(VENUS,OFF),
 DITHER=(ON,0.002222,0.360000,0.000000,0.002222,0.509100,0.000000),
 WINDOW=(2017:007:00:00:00.000,2017:012:23:59:00.000),
 ROLL=(291.159500,0.000000),SEGMENT=(1,5400.000000),PRECEDING=(18695),
 MIN_ACQ=1,MIN_GUIDE=1
"""

In [11]:
len(orig_OR.split("\n\n")), len(targ_atts)

(8, 8)

In [12]:
import re

## Editing OR list with new target atts and offsets to compensate for expected dynamic offsets

In the following code, I loop over the original OR entries and update each so that the TARGET attitude matches the backed out target attitude calculated above, the TARGET_OFFSETs are the inverse of the expected dynamical target offsets, and the ROLL is the target attitude roll as calculated.

In [18]:
for block in orig_OR.split("\n\n"):
    obs_match = re.search("ID=(\d{5}),", block)
    obsid = int(obs_match.group(1))
    # Using the target attitude that doesn't include the dynamic offset bit
    att = targ_atts[targ_atts['obsid'] == obsid][0]
    # and grabbing the dynamic offsets from the file
    dyn = dynamic_offsets[dynamic_offsets['obsid'] == obsid][0]
    block = re.sub("TARGET=(.*?,.*?,.*?),", "TARGET=({:.6f},{:.6f}, {}),".format(att['ra'], att['dec'], '{Venus}'), block)
    # And then flipping the sign on the offsets (I think that makes sense in this application)
    block = re.sub("TARGET_OFFSET=(.*?,.*?),", "TARGET_OFFSET=({:.6f},{:.6f}),".format(
            -1 * (dyn['aca_offset_y'] / 3600.), -1* (dyn['aca_offset_z'] / 3600.)), block)
    block = re.sub("ROLL=(.*?,.*?),", "ROLL=({:.6f},0.000000),".format(att['roll']), block)
    print block
    print "\n"


BEGIN_COMMENT, ID=17439
Window Constraint requirements exist for observation.
  WINDOW=(2017:007:00:00:00,2017:012:23:59:00)
Remarks:
| observing period: 2015 Oct 16 - Nov 07, with preference to Oct 16 - 
20 | instrument: ACIS-I(Venus is optically too bright for ACIS-S) | active 
CCDs: 
only I1 and I3 | readout mode: half array | boresight: Venus should be kept 
within the 1' HEW circle | exposure: 6 x 8 ks = 48 ks 
Note: Current coordinates of 333.395208, -10.236917 are nominal, just to allow 
placement in LTS. (DJP 10/21/16)
END_COMMENT
OBS,
 ID=17439,TARGET=(338.579686,-9.851121, {Venus}),DURATION=(6000.000000),
 PRIORITY=4,SI=ACIS-I,GRATING=NONE,SI_MODE=TE_00C3E,ACA_MODE=DEFAULT,
 TARGET_OFFSET=(-0.001858,0.000197),
 PLANET=(VENUS,OFF),
 DITHER=(ON,0.002222,0.360000,0.000000,0.002222,0.509100,0.000000),
 WINDOW=(2017:009:18:24:47.008,2017:012:23:59:00.000),
 ROLL=(291.014641,0.000000),SEGMENT=(1,5700.000000),PRECEDING=(17764),
 MIN_ACQ=1,MIN_GUIDE=1


BEGIN_COMMENT, ID=18690
Windo